In [ ]:
!pip install -q opencv-python==4.10.0.84
!pip install -q opencv-contrib-python==4.10.0.84
!pip install -q ultralytics
!pip install -q numpy==1.23.5 --force-reinstall # numpyのバージョンを修正し、強制的に再インストール
!pip install -q matplotlib==3.9.0
!pip install -q tqdm==4.66.4

print("\n✓ パッケージのインストールが完了しました")

In [ ]:
# Google Driveをマウント（Colab環境の場合）
import sys
import os
from pathlib import Path

try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Visuable_for_you_tabletennis'
    os.chdir(PROJECT_ROOT)
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'scripts/notebooks'))
    print(f"✓ Google Driveをマウントしました")
    print(f"✓ プロジェクトルート: {PROJECT_ROOT}")
    IN_COLAB = True
    
except ImportError:
    IN_COLAB = False
    notebook_dir = Path(__file__).parent if '__file__' in globals() else Path.cwd()
    sys.path.insert(0, str(notebook_dir))
    print(f"✓ ローカル環境で実行中")
    print(f"✓ 作業ディレクトリ: {Path.cwd()}")

# utilsをインポート
from utils import ColabFileManager, ConfigLoader

# ファイルマネージャーの初期化（プロジェクトルートは自動検出）
fm = ColabFileManager()

print(f"✓ ファイルマネージャー初期化完了")
print(f"  - 検出されたプロジェクトルート: {fm.project_root}")
print(f"  - Colab環境: {fm.is_colab}")

In [ ]:
# 必要なモジュールをインポート
import cv2
import numpy as np
import sys
from pathlib import Path
from typing import Optional, List, Set
from tqdm import tqdm
from IPython.display import HTML
from base64 import b64encode

# プロジェクトのルートディレクトリをパスに追加
sys.path.insert(0, '.')

# プロジェクトモジュールをインポート
from src.detection.table_detector import TableDetector
from src.detection.yolopose_tracker import YOLOPose_Tracker
from src.detection.player_classifier import PlayerClassifier
from src.detection.tracking_exporter import TrackingExporter
from src.visualization.player_classifier_visualizer import PlayerClassifierVisualizer

print("✓ モジュールのインポートが完了しました")

In [ ]:
INPUT_VIDEO = 'data/raw/sample_video_01_short.MOV'
OUTPUT_VIDEO = 'output/player_classification_result.mp4'
CSV_OUTPUT = 'output/player_pose_data.csv' 

In [ ]:
# モデルパス
TABLE_MODEL_PATH = 'models/table_detection/best.pt'
POSE_MODEL_PATH = 'models/pretrained/yolo11l-pose.pt'

# 処理パラメータ
FPS = 30.0              # 処理FPS（GPU利用時は高FPS推奨）
MAX_PLAYERS = 4         # 最大プレイヤー数
MIN_PLAYER_SCORE = 0.3  # プレイヤー判定の最小スコア閾値（0.0-1.0）
MIN_CONSECUTIVE_FRAMES = 30  # CSV出力時に保持する最小連続フレーム数
MAX_FRAME_GAP = 5           # 連続性を判定する際の最大フレーム間隔

print("テストパラメータ:")
print(f"  入力動画: {INPUT_VIDEO}")
print(f"  出力動画: {OUTPUT_VIDEO}")
print(f"  CSV出力: {CSV_OUTPUT}")
print(f"  処理FPS: {FPS}")
print(f"  最大プレイヤー数: {MAX_PLAYERS}")
print(f"  最小スコア閾値: {MIN_PLAYER_SCORE}")
print(f"  最小連続フレーム数: {MIN_CONSECUTIVE_FRAMES}")
print(f"  最大フレーム間隔: {MAX_FRAME_GAP}")

In [ ]:
print(f"\n動画ファイルを開いています: {INPUT_VIDEO}...")
cap = cv2.VideoCapture(INPUT_VIDEO)
if not cap.isOpened():
    print("エラー: 動画ファイルを開けませんでした")

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
video_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"\n入力情報:")
print(f"  解像度: {width}x{height}")
print(f"  動画FPS: {video_fps:.2f}")
print(f"  総フレーム数: {total_frames}")
print(f"  処理FPS: {FPS:.2f}")
print(f"  出力動画FPS: {FPS:.2f} (処理したフレームのみ出力)\n")

# ex1: video_fps=60, FPS=30 -> frame_step=2
#     2フレームに1回処理
# ex2: video_fps=30, FPS=30 -> frame_step=1
#    1フレームに1回処理
frame_step = max(1, round(video_fps / FPS))

table_detector = TableDetector(yolo_model_path=TABLE_MODEL_PATH)
pose_tracker = YOLOPose_Tracker(
    model_path=POSE_MODEL_PATH,
    device = 'cuda'
)
player_classifier = PlayerClassifier(
    max_players=MAX_PLAYERS,
    min_player_score=MIN_PLAYER_SCORE
)
visualizer = PlayerClassifierVisualizer(table_detector, pose_tracker, player_classifier)

print(f"プレイヤー分類設定:")
print(f"  最大プレイヤー数: {MAX_PLAYERS}")
print(f"  最小スコア閾値: {MIN_PLAYER_SCORE:.2f}\n")


fourcc = cv2.VideoWriter_fourcc(*'mp4v')
video_writer = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, FPS, (width, height))
exporter = TrackingExporter()
print(f"CSV出力: {CSV_OUTPUT}\n")

# 卓球台を検出
print("卓球台を検出中...")
table_info = None
max_detection_attempts = 100

for attempt in range(max_detection_attempts):
    ret, frame = cap.read()
    if not ret:
        print("エラー: 動画の終端に達しました")
        cap.release()
        video_writer.release()
        return

    table_info = table_detector.detect_table_from_frame(
        frame, frame_idx=attempt, force_detect=True
    )

    if table_info is not None:
        print(f"✓ 卓球台を検出しました（フレーム {attempt + 1}、信頼度: {table_info.confidence:.2f}）\n")
        break

if table_info is None:
    print(f"エラー: 卓球台を検出できませんでした")
    cap.release()
    video_writer.release()
    return

# 動画を最初に戻す
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

# フレーム処理
frame_count = 0
processed_count = 0
player_ids = set()

print("処理開始...\n")
print(f"  フレーム間隔: {frame_step} ({frame_step}フレームごとに1回処理)")
print(f"  予測処理フレーム数: 約{total_frames // frame_step}フレーム")
print(f"  処理率: {100.0 / frame_step:.1f}%\n")

# プログレスバー付きで処理
pbar = tqdm(total=total_frames, desc="Processing")

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1
        pbar.update(1)

        # 指定FPSで処理（スキップフレームは何もしない）
        if frame_count % frame_step != 0:
            continue

        processed_count += 1

        # 人物を検出・追跡
        if table_info:
            persons = pose_tracker.track_frame_with_table_filter(frame, table_info)
        else:
            persons = pose_tracker.track_frame(frame)

        # プレイヤー分類器を更新
        if table_info and persons:
            player_classifier.update(persons, table_info, frame_count)

        # プレイヤーを分類
        if table_info:
            selected_ids, removed_ids = player_classifier.classify_players()
            player_ids = set(selected_ids)

            if removed_ids:
                pose_tracker.remove_validated_track_ids(removed_ids)

        # CSV出力: プレイヤーとして判定された人物のみを記録
        if player_ids:
            # プレイヤーのみをフィルタリング
            player_persons = [p for p in persons if p.track_id in player_ids]
            if player_persons:
                timestamp = frame_count / video_fps
                exporter.add_frame(frame_count, timestamp, player_persons)

        # 結果を描画
        display_frame = visualizer.draw_results(
            frame, table_info, persons, player_ids
        )
        display_frame = visualizer.draw_candidate_info(
            display_frame, player_ids
        )

        # フレーム情報を表示
        cv2.putText(
            display_frame,
            f"Frame: {frame_count}/{total_frames} (Processed: {processed_count})",
            (10, height - 20),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 255, 255),
            2
        )

        cv2.putText(
            display_frame,
            f"Detected: {len(persons)} persons, Players: {len(player_ids)}",
            (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 255),
            2
        )

        # ビデオに保存（処理したフレームのみ）
        video_writer.write(display_frame)

finally:
    pbar.close()
    cap.release()
    video_writer.release()

    # 最終結果を表示
    print(f"\n✓ 処理完了:")
    print(f"  処理フレーム数: {frame_count}")
    print(f"  実際に処理したフレーム数: {processed_count}")
    print(f"  検出されたプレイヤーID: {sorted(player_ids)}")
    print(f"  候補者数: {len(player_classifier.candidates)}")

    # 候補者詳細情報
    if player_classifier.candidates:
        print(f"\n=== 候補者詳細 ===")
        candidates = []
        for track_id, candidate in player_classifier.candidates.items():
            if candidate.total_frames >= player_classifier.min_tracking_frames:
                score = player_classifier._calculate_player_score(candidate)
                candidates.append((track_id, candidate, score))

        candidates.sort(key=lambda x: x[2], reverse=True)

        for track_id, candidate, score in candidates:
            is_player = track_id in player_ids
            print(f"\nID {track_id} {'[PLAYER]' if is_player else ''}:")
            print(f"  スコア: {score:.3f}")
            print(f"  フレーム数: {candidate.total_frames}")
            print(f"  総運動量: {candidate.total_movement:.1f}")
            print(f"  卓球台付近比率: {candidate.near_table_ratio:.1%}")

    print(f"\n出力ビデオ: {OUTPUT_VIDEO} ({FPS:.1f}fps)")

    # CSV出力
    if exporter:
        # 連続性フィルタリングを適用
        exporter.filter_by_consecutive_frames(
            min_consecutive_frames=MIN_CONSECUTIVE_FRAMES,
            max_frame_gap=MAX_FRAME_GAP
        )

        # プレイヤーの役割情報を作成（すべてのプレイヤーIDに"player"を割り当て）
        player_roles = {track_id: "player" for track_id in player_ids}
        exporter.export_csv(CSV_OUTPUT, player_roles)
        print(f"\nプレイヤー骨格データをCSVに保存しました: {CSV_OUTPUT}")



## 6. 結果の確認

In [ ]:
# 結果動画を表示
def show_video(video_path, width=800):
    """動画をNotebook内に表示"""
    mp4 = open(video_path, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    return HTML(f"""
    <video width={width} controls>
        <source src="{data_url}" type="video/mp4">
    </video>
    """)

if Path(OUTPUT_VIDEO).exists():
    print("結果動画:")
    show_video(OUTPUT_VIDEO)
else:
    print("出力動画が見つかりません")

In [ ]:
# 結果ファイルをダウンロード
from google.colab import files

# 動画ファイルをダウンロード
if Path(OUTPUT_VIDEO).exists():
    print("結果動画をダウンロードしています...")
    files.download(OUTPUT_VIDEO)
    print("✓ 動画のダウンロードが完了しました")
else:
    print("⚠ 出力動画が見つかりません")

# CSVファイルをダウンロード
if Path(CSV_OUTPUT).exists():
    print("\nCSVファイルをダウンロードしています...")
    files.download(CSV_OUTPUT)
    print("✓ CSVのダウンロードが完了しました")
else:
    print("⚠ CSVファイルが見つかりません")